# ISLES'24 Fold 4 → Kaggle handoff — fixed

This CPU-only notebook prepares the Kaggle resume package **without copying the 3+ GiB preprocessed dataset into another Google Drive folder first**.

The previous notebook failed during that duplicate Drive-to-Drive copy. This version streams the existing persistent files directly into one local `/content` tar archive, then uploads that archive as a **private Kaggle Dataset**.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


## 1. Load the latest repository


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')
if REPO.exists(): shutil.rmtree(REPO)
if REPO_ZIP.exists(): REPO_ZIP.unlink()
subprocess.run([
    'curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o',str(REPO_ZIP)
],check=True)
subprocess.run(['unzip','-q',str(REPO_ZIP),'-d','/content'],check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-q','-e','.'],check=True)
print('Repository ready:', REPO)


## 2. Validate sources and local disk


In [ ]:
ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
PRE = ROOT/'nnUNet_preprocessed'/'Dataset504_ISLES2024_NCCT'
FOLD4 = ROOT/'nnUNet_results'/'Dataset504_ISLES2024_NCCT'/'nnUNetTrainer__nnUNetPlans__3d_fullres'/'fold_4'
CHECKPOINT = FOLD4/'checkpoint_latest.pth'
LOCAL = Path('/content/kaggle_handoff_fixed')
LOCAL.mkdir(parents=True,exist_ok=True)

if not PRE.is_dir(): raise FileNotFoundError(PRE)
if not CHECKPOINT.is_file(): raise FileNotFoundError(CHECKPOINT)

usage=shutil.disk_usage('/content')
print('Preprocessed source:', PRE)
print('Fold 4 checkpoint:', CHECKPOINT)
print('Checkpoint MiB:', round(CHECKPOINT.stat().st_size/(1024**2),1))
print('Local free GiB:', round(usage.free/(1024**3),1))
if usage.free < 8*(1024**3):
    raise RuntimeError('Need at least 8 GiB free in /content for the local handoff archive.')


## 3. Build the handoff archive locally

This reads from Drive once and writes directly to local Colab disk. No duplicate 3+ GiB Drive folder is created.


In [ ]:
subprocess.run([
    'python','scripts/make_isles2024_kaggle_resume_bundle.py',
    '--preprocessed-root',str(ROOT/'nnUNet_preprocessed'),
    '--results-root',str(ROOT/'nnUNet_results'),
    '--output-dir',str(LOCAL),
    '--fold','4'
],check=True)

ARCHIVE = LOCAL/'isles2024_fold4_resume.tar'
if not ARCHIVE.is_file(): raise FileNotFoundError(ARCHIVE)
print('Archive ready:', ARCHIVE)
print('Archive GiB:', round(ARCHIVE.stat().st_size/(1024**3),2))


## 4. Authenticate to Kaggle

Upload your `kaggle.json` API token when prompted. This file is tiny and stays only in the temporary Colab runtime.


In [ ]:
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise RuntimeError('Please upload kaggle.json')
KAGGLE_DIR = Path.home()/'.kaggle'
KAGGLE_DIR.mkdir(exist_ok=True)
(KAGGLE_DIR/'kaggle.json').write_bytes(uploaded['kaggle.json'])
os.chmod(KAGGLE_DIR/'kaggle.json',0o600)
subprocess.run(['python','-m','pip','install','-q','kaggle'],check=True)
subprocess.run(['kaggle','config','view'],check=True)


## 5. Upload as a private Kaggle Dataset


In [ ]:
credentials=json.loads((KAGGLE_DIR/'kaggle.json').read_text())
username=credentials['username']
UPLOAD=Path('/content/kaggle_upload_fixed')
if UPLOAD.exists(): shutil.rmtree(UPLOAD)
UPLOAD.mkdir()
shutil.copy2(ARCHIVE,UPLOAD/ARCHIVE.name)
metadata={
    'title':'ISLES2024 NCCT Fold4 Resume - Private',
    'id':f'{username}/isles2024-ncct-fold4-resume',
    'licenses':[{'name':'other'}]
}
(UPLOAD/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))

check=subprocess.run(['kaggle','datasets','files',metadata['id']],capture_output=True,text=True)
if check.returncode==0:
    subprocess.run(['kaggle','datasets','version','-p',str(UPLOAD),'-m','Updated Fold 4 resume state','-r','skip'],check=True)
else:
    subprocess.run(['kaggle','datasets','create','-p',str(UPLOAD),'-r','skip'],check=True)
print('Private Kaggle Dataset:',metadata['id'])


## Next

After the upload completes, create a Kaggle Notebook with GPU enabled, attach the private Dataset `isles2024-ncct-fold4-resume`, and run repository notebook `13_kaggle_resume_isles2024_fold4.ipynb`.
